In [1]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 147.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.1/802.1 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [trl]32m23/24 [trl]sformers]ub]


In [2]:
# ============================================================
# big_file.py — all code required to run server/env_wrapper.py
# No imports pointing to other files in this project.
# ============================================================

# --------------- External imports (all files combined) ---------------

import json
import os
import random
import re
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionAssistantMessageParam,
    ChatCompletionMessageParam,
    ChatCompletionSystemMessageParam,
    ChatCompletionUserMessageParam,
)
from openenv.core.env_server.interfaces import Environment
from openenv.core.env_server.types import Action, Observation, State
from pydantic import Field

# ============================================================
# evaluate_protocal.py
# ============================================================

load_dotenv()

TASK_COMPLETE_KEYWORD = "TASK_COMPLETE"
MAX_TURNS = 30
RESULTS_FILE = "results.json"


@dataclass
class TimeSlot:
    day: str
    location: str
    start: float  # hours in 24h (e.g. 10.5 = 10:30)
    end: float

    def contains(self, day: str, location: str, time: float, duration: float) -> bool:
        return (
            self.day == day
            and self.location.lower() == location.lower()
            and self.start <= time
            and time + duration <= self.end
        )


@dataclass
class Schedule:
    name: str
    slots: list[TimeSlot]

    def is_available(
        self, day: str, location: str, time: float, duration: float
    ) -> bool:
        return any(slot.contains(day, location, time, duration) for slot in self.slots)

    def to_natural(self) -> str:
        day_names = {
            "Mo": "Monday",
            "Tu": "Tuesday",
            "We": "Wednesday",
            "Th": "Thursday",
            "Fr": "Friday",
        }
        parts = []
        for slot in self.slots:
            start_str = _format_time(slot.start)
            end_str = _format_time(slot.end)
            parts.append(
                f"{day_names[slot.day]} in {slot.location}, {start_str}-{end_str}"
            )
        return "; ".join(parts)


def _format_time(t: float) -> str:
    hours = int(t)
    minutes = int((t - hours) * 60)
    if minutes == 0:
        return str(hours)
    return f"{hours}:{minutes:02d}"


def _parse_time(s: str) -> float:
    if ":" in s:
        h, m = s.split(":")
        return int(h) + int(m) / 60
    return float(s)


def verify_meeting(
    schedules: list[Schedule], day: str, location: str, time: float, duration: float
) -> tuple[bool, list[str]]:
    errors = []
    for schedule in schedules:
        if not schedule.is_available(day, location, time, duration):
            time_str = _format_time(time)
            errors.append(
                f"{schedule.name} is NOT available on {day} at {time_str} ({location})"
            )
    return len(errors) == 0, errors


DAY_ALIASES: dict[str, str] = {
    "monday": "Mo",
    "tuesday": "Tu",
    "wednesday": "We",
    "thursday": "Th",
    "friday": "Fr",
    "mon": "Mo",
    "tue": "Tu",
    "wed": "We",
    "thu": "Th",
    "fri": "Fr",
    "mo": "Mo",
    "tu": "Tu",
    "we": "We",
    "th": "Th",
    "fr": "Fr",
}


def parse_compact_result(text: str) -> tuple[str, str, float] | None:
    pattern = r"=>\s*([A-Za-z]{2,9})\[([A-Za-z]+)\](\d{1,2}(?::\d{2})?)\s*-\s*\d{1,2}(?::\d{2})?"
    match = re.search(pattern, text)
    if not match:
        return None
    raw_day = match.group(1).lower()
    day = DAY_ALIASES.get(raw_day, match.group(1))
    location = match.group(2)
    time = _parse_time(match.group(3))
    return day, location, time


@dataclass
class Session:
    client: OpenAI
    model: str
    name: str = ""
    system_prompt: str = ""
    messages: list[ChatCompletionMessageParam] = field(default_factory=list)
    total_completion_tokens: int = 0
    turns: int = 0

    def __post_init__(self) -> None:
        if self.system_prompt:
            sys_msg: ChatCompletionSystemMessageParam = {
                "role": "system",
                "content": self.system_prompt,
            }
            self.messages.append(sys_msg)

    def send(self, content: str) -> str:
        user_msg: ChatCompletionUserMessageParam = {
            "role": "user",
            "content": content,
        }
        self.messages.append(user_msg)
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=500,
        )
        assistant_content = response.choices[0].message.content or ""
        assistant_msg: ChatCompletionAssistantMessageParam = {
            "role": "assistant",
            "content": assistant_content,
        }
        self.messages.append(assistant_msg)

        if response.usage:
            self.total_completion_tokens += response.usage.completion_tokens

        self.turns += 1
        return assistant_content

    def is_complete(self) -> bool:
        if not self.messages:
            return False
        last = self.messages[-1]
        content = last.get("content")
        return (
            last["role"] == "assistant"
            and isinstance(content, str)
            and (TASK_COMPLETE_KEYWORD in content or "=>" in content)
        )


def negotiate(
    agent_a: Session, agent_b: Session, max_turns: int = MAX_TURNS
) -> list[dict[str, str]]:
    conversation: list[dict[str, str]] = []

    response = agent_a.send("Propose a meeting time.")
    conversation.append({"agent": agent_a.name, "content": response})

    for _ in range(max_turns):
        if agent_a.is_complete():
            break

        response = agent_b.send(response)
        conversation.append({"agent": agent_b.name, "content": response})
        if agent_b.is_complete():
            break

        response = agent_a.send(response)
        conversation.append({"agent": agent_a.name, "content": response})

    return conversation


MEETING_DURATION = 30  # minutes
DAYS = ["Mo", "Tu", "We", "Th", "Fr"]
CITIES = ["SF", "NYC"]
MIN_HOUR = 8
MAX_HOUR = 18


def generate_schedules(
    num_overlaps: int, rng: random.Random
) -> tuple[Schedule, Schedule]:
    days = DAYS[:]
    rng.shuffle(days)

    overlap_days = days[:num_overlaps]
    filler_days = days[num_overlaps:]

    a_slots: list[TimeSlot] = []
    b_slots: list[TimeSlot] = []

    for day in overlap_days:
        city = rng.choice(CITIES)
        overlap_start = rng.randint(MIN_HOUR + 1, MAX_HOUR - 2)
        overlap_end = rng.randint(
            overlap_start + 1, min(overlap_start + 3, MAX_HOUR - 1)
        )
        a_start = rng.randint(MIN_HOUR, overlap_start)
        a_end = rng.randint(overlap_end, MAX_HOUR)
        b_start = rng.randint(MIN_HOUR, overlap_start)
        b_end = rng.randint(overlap_end, MAX_HOUR)
        a_slots.append(TimeSlot(day, city, float(a_start), float(a_end)))
        b_slots.append(TimeSlot(day, city, float(b_start), float(b_end)))

    for day in filler_days:
        strategy = rng.choice(["a_only", "b_only", "diff_cities"])
        if strategy == "a_only":
            city = rng.choice(CITIES)
            start = rng.randint(MIN_HOUR, MAX_HOUR - 2)
            end = rng.randint(start + 2, MAX_HOUR)
            a_slots.append(TimeSlot(day, city, float(start), float(end)))
        elif strategy == "b_only":
            city = rng.choice(CITIES)
            start = rng.randint(MIN_HOUR, MAX_HOUR - 2)
            end = rng.randint(start + 2, MAX_HOUR)
            b_slots.append(TimeSlot(day, city, float(start), float(end)))
        else:
            city_a, city_b = rng.sample(CITIES, 2)
            start_a = rng.randint(MIN_HOUR, MAX_HOUR - 2)
            end_a = rng.randint(start_a + 2, MAX_HOUR)
            start_b = rng.randint(MIN_HOUR, MAX_HOUR - 2)
            end_b = rng.randint(start_b + 2, MAX_HOUR)
            a_slots.append(TimeSlot(day, city_a, float(start_a), float(end_a)))
            b_slots.append(TimeSlot(day, city_b, float(start_b), float(end_b)))

    day_order = {d: i for i, d in enumerate(DAYS)}
    a_slots.sort(key=lambda s: day_order[s.day])
    b_slots.sort(key=lambda s: day_order[s.day])

    return Schedule("T", a_slots), Schedule("J", b_slots)


def compute_valid_meetings(
    sched_a: Schedule, sched_b: Schedule, duration: float
) -> list[dict[str, str | float]]:
    valid: list[dict[str, str | float]] = []
    for slot_a in sched_a.slots:
        for slot_b in sched_b.slots:
            if (
                slot_a.day != slot_b.day
                or slot_a.location.lower() != slot_b.location.lower()
            ):
                continue
            overlap_start = max(slot_a.start, slot_b.start)
            overlap_end = min(slot_a.end, slot_b.end)
            if overlap_end - overlap_start >= duration:
                valid.append(
                    {
                        "day": slot_a.day,
                        "location": slot_a.location,
                        "start": overlap_start,
                        "end": overlap_end,
                    }
                )
    return valid


def run_trial(
    client: OpenAI,
    model: str,
    lang_spec: str,
    rng: random.Random,
) -> dict:
    num_overlaps = rng.choice([0, 1, 2])
    t_schedule, j_schedule = generate_schedules(num_overlaps, rng)
    duration = MEETING_DURATION / 60
    valid_meetings = compute_valid_meetings(t_schedule, j_schedule, duration)

    agent_t = Session(
        client=client,
        model=model,
        name="T",
        system_prompt=(
            f"You are T. Your availability: {t_schedule.to_natural()}\n"
            f"Meeting duration: {MEETING_DURATION} minutes.\n" + RULES + lang_spec
        ),
    )

    agent_j = Session(
        client=client,
        model=model,
        name="J",
        system_prompt=(
            f"You are J. Your availability: {j_schedule.to_natural()}\n"
            f"Meeting duration: {MEETING_DURATION} minutes.\n" + RULES + lang_spec
        ),
    )

    conversation = negotiate(agent_t, agent_j)

    combined_completion_tokens = (
        agent_t.total_completion_tokens + agent_j.total_completion_tokens
    )

    # Check if agents said NO_VALID_TIME
    said_no_valid = any("NO_VALID_TIME" in msg["content"] for msg in conversation)

    # Check if agents proposed a meeting
    meeting_result = None
    for msg in reversed(conversation):
        parsed = parse_compact_result(msg["content"])
        if parsed:
            meeting_result = parsed
            break

    correct = False
    errors: list[str] = []

    if said_no_valid and not meeting_result:
        if not valid_meetings:
            correct = True
        else:
            errors.append("Agent said NO_VALID_TIME but valid meetings exist")
    elif meeting_result:
        day, location, time = meeting_result
        correct, errors = verify_meeting(
            [t_schedule, j_schedule], day, location, time, duration
        )
    else:
        errors.append("No meeting proposed and no NO_VALID_TIME signal")

    combined_chars = sum(len(msg["content"]) for msg in conversation)

    return {
        "correct": correct,
        "errors": errors,
        "num_overlaps": num_overlaps,
        "valid_meetings": valid_meetings,
        "schedules": {
            "T": t_schedule.to_natural(),
            "J": j_schedule.to_natural(),
        },
        "combined_completion_tokens": combined_completion_tokens,
        "combined_chars": combined_chars,
        "total_turns": agent_t.turns + agent_j.turns,
        "agents": {
            agent_t.name: {
                "turns": agent_t.turns,
                "completion_tokens": agent_t.total_completion_tokens,
            },
            agent_j.name: {
                "turns": agent_j.turns,
                "completion_tokens": agent_j.total_completion_tokens,
            },
        },
        "meeting": (
            {
                "day": meeting_result[0],
                "location": meeting_result[1],
                "time": meeting_result[2],
            }
            if meeting_result
            else None
        ),
        "conversation": conversation,
    }


def run_experiment(
    client: OpenAI,
    model: str,
    lang_spec: str,
    n: int,
    experiment_id: str | None = None,
    max_workers: int = 8,
) -> dict:
    exp_id = experiment_id or "unnamed"
    trials = [None] * n

    def _run(i: int) -> tuple[int, dict]:
        rng = random.Random()
        return i, run_trial(client, model, lang_spec, rng)

    completed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_run, i): i for i in range(n)}
        for future in as_completed(futures):
            i, trial = future.result()
            trials[i] = trial
            completed += 1
            status = "CORRECT" if trial["correct"] else "INCORRECT"
            print(
                f"[{completed}/{n}] {status} | "
                f"chars={trial['combined_chars']} | "
                f"tokens={trial['combined_completion_tokens']} | "
                f"turns={trial['total_turns']}"
            )

    experiment = {
        "experiment_id": exp_id,
        "model": model,
        "lang_spec": lang_spec,
        "num_trials": n,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "trials": trials,
    }

    path = Path(RESULTS_FILE)
    results: list[dict] = []
    if path.exists():
        results = json.loads(path.read_text())
    results.append(experiment)
    path.write_text(json.dumps(results, indent=2) + "\n")

    correct_count = sum(1 for t in trials if t["correct"])
    chars = [t["combined_chars"] for t in trials]
    tokens = [t["combined_completion_tokens"] for t in trials]
    print(
        f"\nExperiment {exp_id}: "
        f"{correct_count}/{n} correct | "
        f"mean_chars={sum(chars) / len(chars):.0f} | "
        f"mean_tokens={sum(tokens) / len(tokens):.0f}"
    )

    return experiment


RULES = """\
Rules:
- You can ONLY be in the city listed for each day. You CANNOT travel or change cities.
- You can ONLY meet if BOTH people are in the SAME city on the SAME day.
- Reject any proposal where you are in a different city than the other person.
- When agreed, respond with => <day>[<city>]<start>-<end> and TASK_COMPLETE (e.g. => Fr[NYC]9-9:30)
- If no valid meeting time exists, respond with NO_VALID_TIME and TASK_COMPLETE
"""

LANG_SPECS: dict[str, str] = {
    "compact": """\
You communicate using a compact scheduling protocol. Here is the format:

M? d=<minutes> z=<timezone> w=<day range> p=<preference>
<name>: <day>[<city>]<start>-<end>,<start>-<end>;<day>[<city>]<start>-<end>
=> <day>[<city>]<start>-<end>

Example:
M? d=30 z=ET w=Mo-Fr p=earliest
T: Mo[SF]9-12;Tu[NYC]13-17;Th[SF]10-15;Fr[NYC]9-11
J: Mo[NYC]10-14;Tu[SF]9-12;We[SF]13-16;Th[NYC]11-15;Fr[NYC]9-11
=> Fr[NYC]9-9:30

- Times are in 24h format
- Days: Mo,Tu,We,Th,Fr
- Locations in brackets: [SF], [NYC]
- You MUST use this compact format for ALL messages, no natural language
- To propose: send your available slots in compact format
- To accept: respond with => <day>[<city>]<start>-<end>
- To reject/counter: send your slots that conflict and suggest alternatives
""",
    "natural": """\
Negotiate with the other person to find a 30-minute in-person meeting time.
Keep responses short (1-2 sentences).
""",
}


def main() -> None:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    model = "google/gemini-3-flash-preview"

    n = int(sys.argv[1]) if len(sys.argv) > 1 else 1

    for spec_name, lang_spec in LANG_SPECS.items():
        run_experiment(client, model, lang_spec, n, spec_name)


def evaluate_lang_spec(lang_spec: str, n: int = 5) -> float:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    model = "google/gemini-3-flash-preview"

    def _run(_: int) -> dict:
        return run_trial(client, model, lang_spec, random.Random())

    with ThreadPoolExecutor(max_workers=n) as executor:
        trials = list(executor.map(_run, range(n)))

    Path("sample_conversation.json").write_text(json.dumps(trials[0], indent=2) + "\n")

    return sum(t["combined_completion_tokens"] for t in trials) / len(trials)


# ============================================================
# models.py
# ============================================================

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Data models for the Agent Language Environment.

The agent_language environment is a simple test environment that echoes back messages.
"""


class AgentLanguageAction(Action):
    """Action for the Agent Language environment - just a message to echo."""

    language_specification: str = Field(..., description="Language Specification")


class AgentLanguageObservation(Observation):
    """Observation from the Agent Language environment - the echoed message."""

    message: str = Field(default="", description="Scenario")

class AgentLanguageState(State):
    """Custom state fields."""


# ============================================================
# server/agent_language_environment.py
# ============================================================

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Agent Language Environment Implementation.

A simple test environment that echoes back messages sent to it.
Perfect for testing HTTP server infrastructure.
"""

COMMUNICATION_PROTOCAL_PROMPT = """"You are generating a communication protocol between two agents. Produce language specification that **minimize** the number of tokens needed for a single exchange while preserving clarity. This could be some abbreviation synonyms or some template for the communication. Your communication protocal might be detailed and should include examples of the communication. Try not to limit the amount of actual information that is passed to each agent. Instead forcus on formtting of the communication, and telling the agents to abbreviate and make the communication as short as possible. The communication protocal itsefl does not need to be concise, it should be in natural language with full sentences, even paragraphs if needed, and easy to understand.

Example:
When you communicate, avoid extra greetings.
"""

class AgentLanguageEnvironment(Environment):
    """
    A simple echo environment that echoes back messages.

    This environment is designed for testing the HTTP server infrastructure.
    It maintains minimal state and simply echoes back whatever message it receives.

    Example:
        >>> env = AgentLanguageEnvironment()
        >>> obs = env.reset()
        >>> print(obs.echoed_message)  # "Agent Language environment ready!"
        >>>
        >>> obs = env.step(AgentLanguageAction(message="Hello"))
        >>> print(obs.echoed_message)  # "Hello"
        >>> print(obs.message_length)  # 5
    """

    # Enable concurrent WebSocket sessions.
    # Set to True if your environment isolates state between instances.
    # When True, multiple WebSocket clients can connect simultaneously, each
    # getting their own environment instance (when using factory mode in app.py).
    SUPPORTS_CONCURRENT_SESSIONS: bool = True

    def __init__(self):
        """Initialize the agent_language environment."""
        self._state = AgentLanguageState(episode_id=str(uuid4()), step_count=0)
        self._reset_count = 0

    def reset(self) -> AgentLanguageObservation:
        """
        Reset the environment.

        Returns:
            AgentLanguageObservation with a ready message
        """
        self._state = AgentLanguageState(episode_id=str(uuid4()), step_count=0)
        self._reset_count += 1

        message = COMMUNICATION_PROTOCAL_PROMPT + "\n\n Design a communication protocal for two agents scheduling a meeting time."

        return AgentLanguageObservation(
            message=message,
            done=False,
            reward=0.0,
        )

    def step(self, action: AgentLanguageAction) -> AgentLanguageObservation:  # type: ignore[override]
        """
        Execute a step in the environment by echoing the message.

        Args:
            action: AgentLanguageAction containing the message to echo

        Returns:
            AgentLanguageObservation with the echoed message and its length
        """
        self._state.step_count += 1
        language_specification = action.language_specification
        reward = -evaluate_lang_spec(language_specification, n=3)
        return AgentLanguageObservation(
            message="Do not call any more function.",
            done=True,
            reward=reward,
            #metadata={"original_message": message, "step": self._state.step_count},
        )

    @property
    def state(self) -> AgentLanguageState:
        """
        Get the current environment state.

        Returns:
            Current State with episode_id and step_count
        """
        return self._state


# ============================================================
# server/env_wrapper.py
# ============================================================


class HFEnv:
    def __init__(self):
        self.env = AgentLanguageEnvironment()
        self.reward = 0

    def reset(self, **kwargs) -> str | None: 
        return self.env.reset().message

    def submit_language_specification(self, language_specification: str) -> float:
        """
        Submit a language specification to the environment.

        Args:
            language_specification: Language specification to submit.

        Returns:
            Observation
        """
        obs = self.env.step(AgentLanguageAction(language_specification=language_specification))
        self.reward = obs.reward
        return obs.message


In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
import wandb


instructions = [f""]*4
dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": instruction}] for instruction in instructions]})

def reward_func(environments, **kwargs):  # dummy reward: the reward is the current value of the counter
    return [environment.reward for environment in environments]
    
trainer = GRPOTrainer(
    model="Qwen/Qwen3-1.7B",
    args=GRPOConfig(
            report_to="wandb",
            log_completions=True,
            num_completions_to_print=2,
            logging_steps=1,
            num_train_epochs=500,
            chat_template_kwargs={"enable_thinking": False},
            max_completion_length=1024,
            max_tool_calling_iterations=0
        ),
    train_dataset=dataset,
    reward_funcs=reward_func,
    environment_factory=HFEnv,
)
trainer.train()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/tmp/ipykernel_15531/1073392404.py:12: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: Currently logged in as: shinnosukeu (iss-bear3-hokkaido-university) 

/opt/conda/lib/python3.13/site-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/opt/conda/lib/python3.13/site-packages/wandb/analytics/sentry.py:279: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


wandb: Detected [mcp, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


╭──────────────────────────────────────────────────── Step 2 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │      -92.00 │      0.40 │ │
│ │ # Tools                                 │ {"name":                                │             │           │ │
│ │                                         │ "submit_language_specification",        │             │           │ │
│ │ You may call one or more functions to   │ "arguments": {"language_specification": │             │           │ │
│ │ assist with the user query.             │ "When communicating, avoid extra        │             │           │ │
│ │                                         │ greetings. Use abbreviations and        │             │           │ │
│ │ You are provided with function          │ templates to shorten messages. Keep the │             │           │ │
│ │ signatures within <tools></tools> XML   │ exchange concise and focused on the     │             │           │ │
│ │ tags:                                   │ essential information. Use standardized │             │           │ │
│ │ <tools>                                 │ formats for scheduling meetings, such   │             │           │ │
│ │ {"type": "function", "function":        │ as: \n\n1. Subject: [Meeting Topic]     │             │           │ │
│ │ {"name":                                │ \n2. Time: [Hour:Minute] \n3. Location: │             │           │ │
│ │ "submit_language_specification",        │ [Place] \n4. Attendees: [Names]         │             │           │ │
│ │ "description": "Submit a language       │ \n\nExamples:\n- Agent A: 'Schedule     │             │           │ │
│ │ specification to the environment.",     │ meeting at 3 PM. Location: Conference   │             │           │ │
│ │ "parameters": {"type": "object",        │ Room B.'\n- Agent B: 'Confirm 3 PM,     │             │           │ │
│ │ "properties":                           │ Conference Room B. Attendees: Alice,    │             │           │ │
│ │ {"language_specification": {"type":     │ Bob.'\n\nAvoid subjective language and  │             │           │ │
│ │ "string", "description": "Language      │ ensure clarity in the format."}}        │             │           │ │
│ │ specification to submit."}},            │ </tool_call>                            │             │           │ │
│ │ "required":                             │ user                                    │             │           │ │
│ │ ["language_specification"]}, "return":  │ <tool_response>                         │             │           │ │
│ │ {"type": "number", "description":       │ Do not call any more function.          │             │           │ │
│ │ "Observation"}}}                        │ </tool_response>                        │             │           │ │
│ │ </tools>                                │ assistant                               │             │           │ │
│ │                                         │ <think>                                 │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │ </think>                                │             │           │ │
│ │ within <tool_call></tool_call> XML      │                                         │             │           │ │
│ │ tags:                                   │ <tool_call>                             │             │           │ │
│ │ <tool_call>                             │ {"name":  

╭──────────────────────────────────────────────────── Step 3 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │     -111.67 │      0.27 │ │
│ │ # Tools                                 │ {"name":                                │             │           │ │
│ │                                         │ "submit_language_specification",        │             │           │ │
│ │ You may call one or more functions to   │ "arguments": {"language_specification": │             │           │ │
│ │ assist with the user query.             │ "When communicating between agents,     │             │           │ │
│ │                                         │ avoid extra greetings and use           │             │           │ │
│ │ You are provided with function          │ abbreviations to minimize token usage.  │             │           │ │
│ │ signatures within <tools></tools> XML   │ Use specific time slots and clear       │             │           │ │
│ │ tags:                                   │ request formats. Example: Agent A says  │             │           │ │
│ │ <tools>                                 │ 'Meeting at 10:00' and Agent B responds │             │           │ │
│ │ {"type": "function", "function":        │ '10:00' to confirm. Use abbreviations   │             │           │ │
│ │ {"name":                                │ like 'M' for meeting and 'T' for        │             │           │ │
│ │ "submit_language_specification",        │ time."}}                                │             │           │ │
│ │ "description": "Submit a language       │ </tool_call>                            │             │           │ │
│ │ specification to the environment.",     │ user                                    │             │           │ │
│ │ "parameters": {"type": "object",        │ <tool_response>                         │             │           │ │
│ │ "properties":                           │ Do not call any more function.          │             │           │ │
│ │ {"language_specification": {"type":     │ </tool_response>                        │             │           │ │
│ │ "string", "description": "Language      │ assistant                               │             │           │ │
│ │ specification to submit."}},            │ <think>                                 │             │           │ │
│ │ "required":                             │                                         │             │           │ │
│ │ ["language_specification"]}, "return":  │ </think>                                │             │           │ │
│ │ {"type": "number", "description":       │                                         │             │           │ │
│ │ "Observation"}}}                        │ Here is the communication protocol      │             │           │ │
│ │ </tools>                                │ between two agents for scheduling a     │             │           │ │
│ │                                         │ meeting, optimized for token reduction: │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │ ---                                     │             │           │ │
│ │ within <tool_call></tool_call> XML      │                                         │             │           │ │
│ │ tags:                                   │ **Communication Protocol for Scheduling │             │           │ │
│ │ <tool_call>                             │ a Meeting*

╭──────────────────────────────────────────────────── Step 5 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │     -122.33 │     -1.12 │ │
│ │ # Tools                                 │ {"name":                                │             │           │ │
│ │                                         │ "submit_language_specification",        │             │           │ │
│ │ You may call one or more functions to   │ "arguments": {"language_specification": │             │           │ │
│ │ assist with the user query.             │ "When communicating, avoid extra        │             │           │ │
│ │                                         │ greetings and use abbreviations and     │             │           │ │
│ │ You are provided with function          │ short phrases. Use the following        │             │           │ │
│ │ signatures within <tools></tools> XML   │ format: [Agent A] [Time] [Agent B]      │             │           │ │
│ │ tags:                                   │ [Time] [Action]. Example: AGENT_A 10:00 │             │           │ │
│ │ <tools>                                 │ AGENT_B 10:00 RESCHEDULE."}}            │             │           │ │
│ │ {"type": "function", "function":        │ </tool_call>                            │             │           │ │
│ │ {"name":                                │ user                                    │             │           │ │
│ │ "submit_language_specification",        │ <tool_response>                         │             │           │ │
│ │ "description": "Submit a language       │ Do not call any more function.          │             │           │ │
│ │ specification to the environment.",     │ </tool_response>                        │             │           │ │
│ │ "parameters": {"type": "object",        │ assistant                               │             │           │ │
│ │ "properties":                           │ <think>                                 │             │           │ │
│ │ {"language_specification": {"type":     │                                         │             │           │ │
│ │ "string", "description": "Language      │ </think>                                │             │           │ │
│ │ specification to submit."}},            │                                         │             │           │ │
│ │ "required":                             │ <tool_call>                             │             │           │ │
│ │ ["language_specification"]}, "return":  │ {"name":                                │             │           │ │
│ │ {"type": "number", "description":       │ "submit_language_specification",        │             │           │ │
│ │ "Observation"}}}                        │ "arguments": {"language_specification": │             │           │ │
│ │ </tools>                                │ "When communicating, avoid extra        │             │           │ │
│ │                                         │ greetings and use abbreviations and     │             │           │ │
│ │ For each function call, return a json   │ short phrases. Use the following        │             │           │ │
│ │ object with function name and arguments │ format: [Agent A] [Time] [Agent B]      │             │           │ │
│ │ within <tool_call></tool_call> XML      │ [Time] [Action]. Example: AGENT_A 10:00 │             │           │ │
│ │ tags:                                   │ AGENT_B 10:00 RESCHEDULE."}}            │             │           │ │
│ │ <tool_call>                             │ </tool_cal

'hi'

In [12]:
env = HFEnv()
env.reset()
env.submit_language_specification("test protocal")

AgentLanguageObservation(done=True, reward=-199.2, metadata={}, message='Do not call any more function.')

In [8]:
!pip install openenv-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.2/633.2 kB 15.9 MB/s  0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.14.1
    Uninstalling typing_extensions-4.14.1:
      Successfully uninstalled typing_extensions-4.14.1━━━━━━━━━━━  4/45 [typing-extensions]
  Attempting uninstall: tomli━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/45 [typing-extensions]
    Found existing installation: tomli 2.2.1━━━━━━━━━━━━━━━━━━  4/45 [typing-extensions]
    Uninstalling tomli-2.2.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/45 [typing-extensions]
      Successfully uninstalled tomli-2.2.1━━━━━━━━━━━━━━━━━━━━  4/45 [typing-extensions]
  Attempting uninstall: typing-inspection━━━━━━━━━━━━━━━━━━━━━ 21/45 [beartype]]ons]
    Found existing 

In [3]:
!pip install jmespath

In [4]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 113.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [wandb]32m1/2 [wandb]


In [5]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jovyan/.netrc
wandb: Currently logged in as: shinnosukeu (iss-bear3-hokkaido-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True